# CardioSense Phase 1 — Pipeline 3: Chest X-ray Analysis

**NIH ChestX-ray14 → patient-level split → DenseNet121 transfer learning → Grad-CAM**

Run `00_colab_setup.ipynb` first, including section 7c (the ChestX-ray14 download).

**This notebook needs a GPU.** `Runtime → Change runtime type → GPU`.

### Sections
1. Environment Setup · 2. Imports · 3. Configuration · 4. Dataset Verification
5. Exploratory Data Analysis · 6. Preprocessing & Augmentation · 7. Patient-Level Split
8. Baselines · 9. Model Definition · 10. Training · 11. Validation · 12. Evaluation
13. Explainability · 14. *(calibration — see note)* · 15. Error Analysis
16. Save Model · 17. Save Results · 18. Example Inference

### The two things that decide whether this pipeline is honest

**1. The split is by PATIENT, never by image.** A patient contributes several
films — same chest, same body habitus, same hardware. An image-level split puts
the same chest on both sides and inflates AUC by several points. It is invisible
in the metrics: the model just looks better than it is. §7 asserts zero patient
overlap rather than trusting it.

**2. Accuracy is not the metric.** At the real ~2.5% prevalence, predicting "no
cardiomegaly" for every image scores ~97.5% accuracy and finds nothing. The
majority-class baseline in §8 exists specifically to put that number in the
results table where nobody can mistake it for success. **PR-AUC is the headline**,
and its chance level is the prevalence, reported beside it every time.

### Experiments
| ID | Experiment | Question |
|---|---|---|
| X-A | Majority class + pixel-feature LogReg | What floor must the CNN clear? |
| X-B | DenseNet121 transfer learning | Does it clear it? |

**On disconnects:** every epoch writes `last.pt` — including the staged-unfreeze
state. Re-run the training cell and it resumes.

## 1. Environment Setup

In [ ]:
import os, sys, subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)

    REPO_DIR = Path("/content/CardioSense")
    if not REPO_DIR.exists():
        raise RuntimeError("Repo not found. Run 00_colab_setup.ipynb first.")
    os.chdir(REPO_DIR)

    os.environ["CARDIOSENSE_DATA_ROOT"] = "/content/drive/MyDrive/CardioSense/data"
    subprocess.run(["pip", "install", "-q", "-r", "requirements-colab.txt"], check=False)
    subprocess.run(["pip", "install", "-q", "-e", "."], check=False)
else:
    here = Path.cwd()
    while not (here / "pyproject.toml").exists() and here != here.parent:
        here = here.parent
    os.chdir(here)

print("Working directory:", Path.cwd())
print("Data root        :", os.environ.get("CARDIOSENSE_DATA_ROOT", "<repo>/data"))

## 2. Imports

In [ ]:
import json
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from IPython.display import Image as ShowImage, display

from cardiosense.common.config import load_config
from cardiosense.common.env import get_device, print_environment
from cardiosense.common.experiment import ExperimentTracker, load_experiment_log
from cardiosense.common.io_utils import save_json, save_pickle
from cardiosense.common.paths import PATHS
from cardiosense.common.plots import plot_class_distribution, plot_training_curves
from cardiosense.common.seeding import set_seed
from cardiosense.common.training import count_parameters

from cardiosense.xray.data import (
    build_target, describe_dataset, load_metadata, resolve_nih_root,
    split_by_patient, verify_dataset,
)
from cardiosense.xray.preprocessing import build_transforms, denormalize
from cardiosense.xray.dataset import ChestXrayDataset, build_dataloaders, compute_pos_weight
from cardiosense.xray.baseline import (
    build_feature_matrix, majority_class_baseline, train_pixel_baseline,
)
from cardiosense.xray.models import build_model, model_summary
from cardiosense.xray.trainer import train_model
from cardiosense.xray import evaluate as ev
from cardiosense.xray.explain import run_gradcam_analysis
from cardiosense.xray.predict import XrayPredictor

pd.set_option("display.width", 150)
pd.set_option("display.max_columns", 40)
_ = print_environment()

## 3. Configuration

In [ ]:
cfg = load_config("xray")

SEED = int(cfg.seed)
set_seed(SEED, strict=bool(cfg.get("strict_determinism", False)))
device = get_device()

RESULTS = PATHS.root / cfg.output.results_dir
MODELS = PATHS.root / cfg.output.models_dir
CHECKPOINTS = MODELS / "checkpoints"
for path in (RESULTS, MODELS, CHECKPOINTS):
    path.mkdir(parents=True, exist_ok=True)

print(f"target        : {cfg.dataset.target_label} (binary)")
print(f"view filter   : {cfg.dataset.filter_view} only")
print(f"negative ratio: {cfg.dataset.negative_ratio} negatives per positive")
print(f"split         : {cfg.split.strategy}, PATIENT-level")
print(f"image size    : {cfg.preprocessing.image_size}px, ImageNet normalisation")
print(f"model         : DenseNet121 (pretrained={cfg.model.pretrained}), "
      f"freeze {cfg.model.freeze_backbone_epochs} epoch(s), "
      f"unfreeze from {cfg.model.unfreeze_from_block}")
print(f"imbalance     : {cfg.class_imbalance.method}")
print(f"headline metric: {cfg.evaluation.primary_metric}")
print(f"device        : {device}")

## 4. Dataset Verification

Both filenames are accepted: the Kaggle mirror ships `Data_Entry_2017.csv`, the
NIH v2020 release uses `Data_Entry_2017_v2020.csv`.

In [ ]:
root = resolve_nih_root(cfg)
print(json.dumps(verify_dataset(cfg, root), indent=2))

images_dir = root / cfg.dataset.images_dir
frame = load_metadata(cfg, root)
frame.head()

## 5. Target Extraction & EDA

The target is a **substring test on the released labels** — nothing is invented.
`Finding Labels` is pipe-separated, so `Cardiomegaly|Effusion` is positive for our
target and also carries a co-morbidity.

**PA views only.** AP films are portable studies on supine, sicker patients, and
they geometrically magnify the cardiac silhouette. Mixing views hands the model a
shortcut — "predict cardiomegaly whenever the film looks like a portable AP" —
that has nothing to do with heart size.

In [ ]:
frame, target_report = build_target(frame, cfg)
save_json(target_report, RESULTS / "target_report.json")

print(f"prevalence, all views      : {target_report['prevalence_all_views']:.4f}")
print(f"prevalence, {cfg.dataset.filter_view} only         : "
      f"{target_report['prevalence_after_view_filter']:.4f}")
print(f"after negative subsampling : {target_report['final_prevalence']:.4f}")
print(f"\nfinal: {target_report['positives']} positive of {target_report['rows_out']} images, "
      f"{target_report['n_patients']} patients")

if "negative_subsampling" in target_report:
    print(f"\nNOTE: {target_report['negative_subsampling']['note']}")

In [ ]:
description = describe_dataset(frame, cfg)
save_json(description, RESULTS / "dataset_description.json")

plot_class_distribution(
    {"No cardiomegaly": int((frame.target == 0).sum()),
     "Cardiomegaly": int((frame.target == 1).sum())},
    RESULTS / "class_distribution.png", title="Target distribution after filtering",
)
display(ShowImage(filename=str(RESULTS / "class_distribution.png")))

print(f"images per patient: mean {description['images_per_patient']['mean']}, "
      f"max {description['images_per_patient']['max']}")
print("\nThat maximum is exactly why the split must be by patient, not by image.\n")

print("Findings that co-occur with the target:")
pd.Series(description["co_occurring_with_target"]).head(10).to_frame("count")

In [ ]:
# Look at some actual images: positive vs negative.
from PIL import Image as PILImage

fig, axes = plt.subplots(2, 4, figsize=(14, 7))
for row, (label, title) in enumerate([(1, "Cardiomegaly"), (0, "No cardiomegaly")]):
    sample = frame[frame.target == label].sample(4, random_state=SEED)
    for ax, (_, record) in zip(axes[row], sample.iterrows()):
        with PILImage.open(images_dir / record[cfg.dataset.image_column]) as handle:
            ax.imshow(handle.convert("L"), cmap="gray")
        ax.set_title(f"{title}\n{record[cfg.dataset.label_column][:40]}", fontsize=8)
        ax.axis("off")
plt.suptitle("Sample radiographs by target class")
plt.tight_layout()
plt.show()

## 6. Preprocessing & Augmentation

Two pipelines, and the difference is the point: **train gets randomness, val and
test get none.** Random transforms on validation make the monitored metric noisy,
so early stopping picks a checkpoint by luck; on test they make the reported
result irreproducible.

| Augmentation | On? | Why |
|---|---|---|
| **Horizontal flip** | **NO** | Mirroring moves the heart to the right side of the chest. That is *dextrocardia* — a real congenital condition. Training with flips teaches the model a right-sided heart is normal and destroys the left-right asymmetry cardiomegaly assessment depends on. The code **raises an error** if this is enabled |
| Vertical flip | NO | An upside-down chest film does not occur |
| Rotation ±7° | yes | Real patient-positioning variation is small |
| Translation 5% | yes | Framing varies a little |
| Brightness/contrast ±10% | yes | Exposure and detector differences between machines |
| RandomResizedCrop 0.9–1.0 | yes | The usual 0.08–1.0 default would crop the heart out entirely and still label it "cardiomegaly" |

Normalisation uses **ImageNet statistics**, because the pretrained DenseNet121
filters were fitted under exactly those.

In [ ]:
transforms = build_transforms(cfg)
print("train:", transforms["train"])
print("\neval (val + test — note the absence of anything random):")
print(transforms["val"])

In [ ]:
# What the augmentation actually does to one image.
sample_path = images_dir / frame.iloc[0][cfg.dataset.image_column]
with PILImage.open(sample_path) as handle:
    original = handle.convert("L")

fig, axes = plt.subplots(1, 5, figsize=(16, 3.6))
axes[0].imshow(original, cmap="gray")
axes[0].set_title("original", fontsize=9)
for i, ax in enumerate(axes[1:], start=1):
    torch.manual_seed(i)
    augmented = denormalize(transforms["train"](original), cfg).numpy().mean(axis=0)
    ax.imshow(augmented, cmap="gray")
    ax.set_title(f"augmented #{i}", fontsize=9)
for ax in axes:
    ax.axis("off")
plt.suptitle("Training augmentation — note the heart stays on the same side in every one")
plt.tight_layout()
plt.show()

## 7. Patient-Level Split

**The most important cell in this notebook.**

A patient contributes several films. Splitting by image puts the same chest in
train and test — same body habitus, same implanted hardware, same scanner — and
inflates AUC by several points while looking completely normal in the metrics.

`split_by_patient` uses NIH's own patient-disjoint lists where available, carves
validation out of train-val **by patient**, and then *asserts* that the patient-ID
intersection between every pair of splits is empty. If it were not, the function
raises rather than returning a quietly corrupted split.

In [ ]:
splits = split_by_patient(frame, cfg, root)
save_json(splits["summary"], RESULTS / "split_summary.json")

print("patient overlap (must be all zero):", splits["summary"]["patient_overlap"])
pd.DataFrame(splits["summary"]["splits"]).T

In [ ]:
y_train = splits["train"].target.to_numpy()
y_val = splits["val"].target.to_numpy()
y_test = splits["test"].target.to_numpy()

# Patient-level splitting cannot stratify perfectly, because patients contribute
# different numbers of images. Check how far apart the split prevalences are.
prevalences = {name: splits["summary"]["splits"][name]["prevalence"]
               for name in ("train", "val", "test")}
print("prevalence per split:", prevalences)
spread = max(prevalences.values()) - min(prevalences.values())
print(f"spread: {spread:.4f}")
if spread > 0.05:
    print("\nNote this when comparing metrics ACROSS splits — a PR-AUC difference "
          "partly reflects the different chance levels, not only the model.")

## 8. Baselines — Experiment X-A

**The majority-class baseline is the point of this section.** It predicts "no
cardiomegaly" for every image. Look at its accuracy, then look at its recall.

The pixel-feature baseline is the floor DenseNet121 must clear. If a 32×32
thumbnail plus an intensity histogram gets most of the way there, the task is
being solved by global exposure and body size rather than cardiac silhouette
shape — and the CNN result deserves scepticism.

In [ ]:
experiments = {}

majority = majority_class_baseline(y_train, y_test)
experiments["majority_class"] = {
    "metrics": {"at_tuned_threshold": majority["metrics"]},
    "parameters": 1, "train_seconds": 0.0,
}

print(f"\naccuracy : {majority['metrics']['accuracy']:.4f}   <- looks great")
print(f"recall   : {majority['metrics']['recall']:.4f}   <- finds nothing")
print(f"TP       : {int(majority['metrics']['tp'])} of {int(y_test.sum())} positive cases")
print("\nThis single row is why accuracy is not the headline metric for this pipeline.")

In [ ]:
start = time.time()
X_train = build_feature_matrix(splits["train"], images_dir, cfg)
X_val = build_feature_matrix(splits["val"], images_dir, cfg)
X_test = build_feature_matrix(splits["test"], images_dir, cfg)

pixel = train_pixel_baseline(X_train, y_train, X_val, cfg, X_test=X_test)
threshold_pixel, _ = ev.tune_threshold(y_val, pixel["val_prob"], cfg)
metrics_pixel = ev.evaluate_binary(y_test, pixel["test_prob"], threshold_pixel, cfg,
                                   split_name="test/pixel_baseline")

experiments["logreg_pixel_features"] = {
    "metrics": metrics_pixel,
    "parameters": pixel["parameters"],
    "train_seconds": round(time.time() - start, 2),
}
save_pickle(pixel["model"], MODELS / "xray_baseline.pkl")
print(f"\npixel baseline PR-AUC {metrics_pixel['at_tuned_threshold']['pr_auc']:.4f} "
      f"(chance {metrics_pixel['prevalence']:.4f})")

## 9. Model Definition — DenseNet121 (Experiment X-B)

**Why DenseNet121:** it is the standard for ChestX-ray14 (CheXNet and successors),
so results are comparable. Dense connectivity means every layer sees all preceding
feature maps, so fine edge detail from early layers survives to the head — and a
cardiac border is a large-scale shape defined by fine edges. It is also small for
its accuracy: ~7 M parameters against ResNet-50's ~25 M.

**Pretrained weights:** ImageNet. Chest X-rays look nothing like ImageNet photos,
and transfer still works, because early layers learn edge, blob and texture
detectors that are close to universal. Only the later semantic layers are
domain-specific — which is exactly the part we retrain.

**Head:** the 1000-way ImageNet classifier is discarded and replaced with dropout
+ one linear unit emitting a single logit. One logit, not two: `BCEWithLogitsLoss`
on one logit is equivalent to, and more stable than, a 2-way softmax.

**Two-stage fine-tuning:**
1. *Head only* — the backbone is frozen. The new head is randomly initialised, so
   its early gradients are large and noisy; letting those propagate into
   pretrained weights destroys the features that make transfer work.
2. *Partial fine-tune* — unfreeze from `denseblock3` onward at a 6× smaller
   learning rate. Early blocks stay frozen: their edge detectors are already
   right, and retraining them on ~10k images mostly overfits.

In [ ]:
loaders = build_dataloaders(splits, images_dir, transforms, cfg)

pos_weight = None
if str(cfg.class_imbalance.method) == "pos_weight":
    pos_weight = compute_pos_weight(y_train)
    print(f"pos_weight = {float(pos_weight):.2f}")
    print("Each positive image contributes as much to the loss as that many negatives.")
    print("\nWhy only this and not a weighted sampler as well: stacking both applies the")
    print("correction twice and produces wildly over-confident probabilities, which would")
    print("poison the confidence-aware fusion planned for Phase 2.\n")

model = build_model(cfg).to(device)
architecture = model_summary(model, input_shape=(2, 3, cfg.preprocessing.image_size,
                                                 cfg.preprocessing.image_size))
print(json.dumps(architecture, indent=2))

## 10. Training

Monitored metric is **`val_pr_auc`**, not loss and certainly not accuracy. At this
prevalence, loss and accuracy can both improve while the model gets worse at the
only thing that matters — identifying the positive cases.

**If Colab disconnects, re-run this cell.** The checkpoint stores the staged-
unfreeze state too, so resuming past the unfreeze epoch restores the unfrozen
backbone rather than silently continuing with it frozen.

In [ ]:
training_result = train_model(model, loaders, cfg, device, CHECKPOINTS, pos_weight=pos_weight)

print(f"\nbest {training_result['monitor']} = {training_result['best_value']:.4f} "
      f"at epoch {training_result['best_epoch']}")
print(f"epochs run : {training_result['epochs_run']} "
      f"(resumed from epoch {training_result['resumed_from']})")
print(f"time       : {training_result['total_seconds']:.0f}s "
      f"({training_result['seconds_per_epoch']:.1f}s/epoch)")
if training_result.get("unfreeze"):
    print(f"unfrozen   : {training_result['unfreeze']['trainable_blocks']} "
          f"({training_result['unfreeze']['trainable_fraction']:.1%} of parameters)")

In [ ]:
plot_training_curves(
    training_result["history"], RESULTS / "training_curve.png",
    loss_keys=("train_loss", "val_loss"),
    metric_keys=("val_pr_auc", "val_roc_auc", "val_recall"),
    best_epoch=training_result["best_epoch"],
    title="DenseNet121 — training history",
)
display(ShowImage(filename=str(RESULTS / "training_curve.png")))

# Look for the stage transition: unfreezing usually produces a visible change in
# both the loss slope and the validation metric.
pd.DataFrame(training_result["history"]).round(4)

## 11. Validation — operating threshold

In [ ]:
use_amp = bool(cfg.training.get("amp", True)) and device.type == "cuda"

val_prob, val_true, _ = ev.predict_probabilities(model, loaders["val"], device, use_amp)
threshold, threshold_info = ev.tune_threshold(val_true, val_prob, cfg)
save_json(threshold_info, RESULTS / "threshold.json")

print(json.dumps(threshold_info, indent=2))
print("\nThe default 0.5 is almost always wrong after pos_weight training: the loss")
print("weighting deliberately shifts the logit scale, so the operating point moves too.")

## 12. Evaluation — test split, scored once

Metric hierarchy, in order:

1. **PR-AUC** — precision and recall both ignore true negatives, so this measures
   performance on the class we care about. Chance level = prevalence, always
   reported beside it.
2. **ROC-AUC** — for comparability with published ChestX-ray14 results.
3. **Recall** — a missed cardiomegaly is the costly error.
4. **Accuracy** — last, and only next to the majority-class row.

In [ ]:
test_prob, test_true, test_idx = ev.predict_probabilities(model, loaders["test"], device, use_amp)
order = np.argsort(test_idx)
test_prob, test_true = test_prob[order], test_true[order]

test_metrics = ev.evaluate_binary(test_true, test_prob, threshold, cfg, split_name="test")
experiments["densenet121"] = {
    "metrics": test_metrics,
    "parameters": count_parameters(model, trainable_only=False),
    "train_seconds": training_result["total_seconds"],
}

tuned = test_metrics["at_tuned_threshold"]
ci = test_metrics.get("confidence_intervals_95pct", {})
print(f"PR-AUC  {tuned['pr_auc']:.4f}  (chance = prevalence {test_metrics['prevalence']:.4f}, "
      f"lift {test_metrics['pr_auc_lift_over_chance']:.1f}x)")
if ci:
    print(f"        95% CI [{ci['pr_auc']['lower']:.3f}, {ci['pr_auc']['upper']:.3f}]")
print(f"ROC-AUC {tuned['roc_auc']:.4f}")
if ci:
    print(f"        95% CI [{ci['roc_auc']['lower']:.3f}, {ci['roc_auc']['upper']:.3f}]")
print(f"\nAlways-negative would score accuracy {test_metrics['accuracy_of_always_negative']:.4f} "
      f"and find 0 of {test_metrics['n_positive']} cases.")

pd.DataFrame({
    f"threshold={threshold:.2f}": {k: tuned[k] for k in
        ("pr_auc", "roc_auc", "precision", "recall", "specificity", "f1", "accuracy")},
    "threshold=0.50": {k: test_metrics["at_default_threshold_0.5"][k] for k in
        ("pr_auc", "roc_auc", "precision", "recall", "specificity", "f1", "accuracy")},
}).round(4)

In [ ]:
ev.plot_evaluation_figures(
    test_true, test_prob, threshold, RESULTS, model_name="DenseNet121",
    extra_curves={"Pixel-feature LogReg": (y_test, pixel["test_prob"])},
)
for name in ["confusion_matrix.png", "roc_curve.png", "pr_curve.png"]:
    display(ShowImage(filename=str(RESULTS / name)))

In [ ]:
# Did DenseNet121 actually beat the baselines? The honest answer is sometimes no.
comparison = ev.build_comparison_table(experiments)
comparison.to_csv(RESULTS / "model_comparison.csv", index=False)
display(comparison)

recommendation = ev.recommend_model(experiments)
save_json(recommendation, RESULTS / "model_recommendation.json")
print(f"\nRECOMMENDED: {recommendation['recommended']}")
for reason in recommendation["reasoning"]:
    print("  -", reason)
if "warning" in recommendation:
    print("\nWARNING:", recommendation["warning"])

print("\nRead the accuracy column against the majority_class row before quoting it "
      "anywhere.")

## 14. Calibration — deliberately not done here

Like the ECG pipeline, no calibrator is fitted for this modality in Phase 1, and
for an additional reason specific to X-ray: **negative subsampling has changed the
prevalence.** The model's probabilities are conditioned on the sampled prior, not
the population one, so calibrating them against this test split would produce a
calibrator that is only valid for the sampled distribution.

Phase 2 needs to do two things in order: correct for the sampling prior, then
calibrate. Both are recorded in `xray_config.json` so the correction is possible.
Until then, the probability is a **ranking score, not a frequency claim**, and
`predict.py` says so in its output.

## 15. Error Analysis

In [ ]:
errors = ev.export_errors(splits["test"], test_true, test_prob, threshold, cfg,
                          RESULTS / "errors", prefix="test")

print(f"{errors['n_errors']} errors of {errors['n_images']} ({errors['error_rate']:.1%})")
print(f"breakdown: {errors['counts']}")

if "false_positive_findings" in errors:
    print("\nFindings present on images the model wrongly flagged:")
    display(pd.Series(errors["false_positive_findings"]).to_frame("count").head(8))
    print(errors["false_positive_findings_note"])
    print("\nIf one finding dominates this list, the model may be keying on it rather "
          "than on heart size.")

In [ ]:
# Where do the errors sit relative to the threshold?
fig, ax = plt.subplots(figsize=(11, 4))
y_pred_test = (test_prob >= threshold).astype(int)
rng = np.random.default_rng(0)
for label, mask, colour in [
    ("correct", y_pred_test == test_true, "tab:green"),
    ("false negative", (test_true == 1) & (y_pred_test == 0), "tab:red"),
    ("false positive", (test_true == 0) & (y_pred_test == 1), "tab:orange"),
]:
    if mask.sum():
        ax.scatter(test_prob[mask], rng.normal(0, 0.04, int(mask.sum())),
                   alpha=0.6, s=35, color=colour, label=f"{label} (n={int(mask.sum())})")
ax.axvline(threshold, color="k", ls="--", label=f"threshold = {threshold:.2f}")
ax.set_xlabel("predicted probability of cardiomegaly")
ax.set_yticks([])
ax.legend(fontsize=9)
ax.set_title("Test predictions — errors far from the threshold are the ones to inspect")
plt.tight_layout()
plt.show()

## 13. Explainability — Grad-CAM

Grad-CAM weights each feature map of the last convolutional block by its
spatially-averaged gradient, sums them, rectifies, and upsamples. It answers
"which regions raised this score?".

**What it does NOT establish** — and this caveat has to survive into the report,
because a red blob over a heart is extraordinarily convincing and proves far less
than it appears to:

1. **It shows where gradient mass concentrated, not why a diagnosis is correct.**
   A hot cardiac silhouette is *consistent with* the model measuring heart size.
   It does not demonstrate it, and it is not evidence of medical reasoning.
2. **It is coarse.** A 7×7 grid upsampled to 224×224 means each cell covers ~32×32
   pixels. The apparent precision is a bilinear interpolation artefact.
3. **A plausible map can accompany a wrong prediction**, and an odd map a correct
   one. Both appear in the figures below — which is why false negatives are
   included, not just successes.
4. **It cannot detect shortcut learning by itself.** If the model keys on a
   pacemaker or a portable-film marker, the map may still land near the heart,
   because those things are near the heart.

In [ ]:
n_per_category = max(1, int(cfg.explainability.n_examples) // 4)
selection = ev.select_error_examples(test_true, test_prob, threshold,
                                     n_per_category=n_per_category)
print("selected cases:", {k: len(v) for k, v in selection.items()})

gradcam_dataset = ChestXrayDataset(
    splits["test"], images_dir, transforms["test"],
    image_column=str(cfg.dataset.image_column), return_index=False,
)
gradcam_summary = run_gradcam_analysis(
    model, splits["test"].reset_index(drop=True), gradcam_dataset, selection,
    test_true, test_prob, threshold, cfg, device, RESULTS / "gradcam",
)

pd.DataFrame(gradcam_summary["cases"])[
    ["case_type", "image", "probability", "true_label", "mass_lower_half",
     "mass_central_third"]
]

In [ ]:
# Show one of each case type. The FN is the informative one: where did the model
# look when it MISSED a cardiomegaly?
shown = set()
for record in gradcam_summary["cases"]:
    if record["case_type"] in shown:
        continue
    shown.add(record["case_type"])
    print(f"--- {record['case_type']}: p = {record['probability']}, "
          f"true = {record['true_label']} ---")
    display(ShowImage(filename=str(RESULTS / "gradcam" / record["figure"])))

## 16. Save Model

In [ ]:
model_path = MODELS / cfg.output.model_file
torch.save({"model_state": model.state_dict(), "model_name": "densenet121",
            "threshold": float(threshold)}, model_path)

inference_config = {
    "model_version": str(cfg.output.model_version),
    "model_name": "densenet121",
    "n_classes": int(cfg.model.n_classes),
    "dropout": float(cfg.model.dropout),
    "target_label": str(cfg.dataset.target_label),
    "label_mapping": {"0": "no cardiomegaly", "1": "cardiomegaly"},
    "threshold": round(float(threshold), 4),
    "image_size": int(cfg.preprocessing.image_size),
    "normalize_mean": list(cfg.preprocessing.normalize_mean),
    "normalize_std": list(cfg.preprocessing.normalize_std),
    "to_rgb": bool(cfg.preprocessing.to_rgb),
    "view_filter": cfg.dataset.filter_view,
    "training_prevalence": round(float(y_train.mean()), 5),
    "negative_subsampling_ratio": cfg.dataset.negative_ratio,
    "prevalence_warning": (
        "Probabilities are conditioned on the sampled training prevalence, which differs "
        "from the population prevalence because negatives were subsampled. Correct for "
        "the prior before reading these as population risk."
    ),
}
config_path = save_json(inference_config, MODELS / cfg.output.config_file)

metadata_path = save_json({
    **inference_config,
    "created_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
    "modality": "xray",
    "dataset": {"name": str(cfg.dataset.name), "root": str(root)},
    "architecture": architecture,
    "split_summary": splits["summary"],
    "target_report": target_report,
    "training": training_result,
    "test_metrics": test_metrics,
    "threshold_info": threshold_info,
    "recommendation": recommendation,
    "class_imbalance": cfg.class_imbalance.to_dict(),
    "augmentation": cfg.augmentation.to_dict(),
    "training_config": cfg.to_dict(),
}, MODELS / cfg.output.metadata_file)

for path in (model_path, config_path, metadata_path):
    print(f"  {path.name:<26} {path.stat().st_size / 1024:>9.1f} KB")

## 17. Save Results

In [ ]:
save_json({
    "model": "densenet121",
    "test": test_metrics,
    "model_comparison": comparison.to_dict(orient="records"),
    "recommendation": recommendation,
    "split": splits["summary"],
    "threshold": threshold_info,
    "training": training_result,
}, RESULTS / "metrics.json")

with ExperimentTracker("xray_notebook", modality="xray", config=cfg,
                       primary_metric="pr_auc") as run:
    run.log_params({
        "model": "densenet121",
        "pretrained": bool(cfg.model.pretrained),
        "batch_size": int(cfg.training.batch_size),
        "learning_rate": float(cfg.training.learning_rate),
        "epochs": int(cfg.training.epochs),
        "parameters": architecture["total"],
        "class_imbalance": str(cfg.class_imbalance.method),
        "n_train": splits["summary"]["splits"]["train"]["n_images"],
    })
    run.log_metrics({k: v for k, v in tuned.items() if isinstance(v, (int, float))},
                    split="test")
    run.log_history(training_result["history"])
    run.set_best_epoch(int(training_result["best_epoch"]))
    run.log_artifact("model", model_path)

print("\nfiles written:")
for path in sorted(RESULTS.rglob("*")):
    if path.is_file():
        print("  ", path.relative_to(RESULTS))

## 18. Example Inference

Loads the saved artifacts from scratch and runs on an image file, the way a Phase
2 caller would.

In [ ]:
predictor = XrayPredictor.load()

test_image = images_dir / splits["test"].iloc[0][cfg.dataset.image_column]
result = predictor.predict_image(test_image)

print(json.dumps({k: v for k, v in result.items() if k != "notes"}, indent=2))
print(f"\nactual label: {int(splits['test'].iloc[0].target)}")
print("\nnotes:")
for key, value in result["notes"].items():
    print(f"  {key}: {value}")

In [ ]:
# Inference with a Grad-CAM figure, in one call.
explained = predictor.explain_image(test_image, RESULTS / "gradcam" / "inference_example.png")
display(ShowImage(filename=explained["gradcam"]))
print(f"probability {explained['probability']} at threshold {explained['threshold']}")

## Phase 1 X-ray checklist

| Item | Where |
|---|---|
| Dataset verified | §4 |
| Target selected | §5 — substring test on released labels, PA views only |
| Patient-level split | §7 — overlap asserted zero |
| Image preprocessing | §6 |
| Augmentation | §6 — train only, no horizontal flip (enforced) |
| Baseline | §8 (X-A) — majority class + pixel features |
| DenseNet121 | §9–10 (X-B) — two-stage fine-tuning |
| Evaluation | §12 — PR-AUC headline, accuracy contextualised |
| Error analysis | §15 |
| Grad-CAM | §13 — with limitations stated |
| Saved model | §16 |
| Inference script | §18 · `src/cardiosense/xray/predict.py` |

## Phase 1 is now complete

All three pipelines are trained, evaluated, explained and saved:

| Pipeline | Notebook | Headline metric | Explainability |
|---|---|---|---|
| Clinical | `01_clinical_training.ipynb` | ROC-AUC + Brier | SHAP |
| ECG | `02_ecg_training.ipynb` | Macro ROC-AUC | Integrated Gradients |
| X-ray | `03_xray_training.ipynb` | PR-AUC | Grad-CAM |

Cross-modality comparison tables live in `docs/phase1_results.md`; every run is
recorded in `results/experiments/experiment_log.csv`.

> Multimodal fusion, RAG, clinical report generation and dashboard integration are
> intentionally outside Phase 1.

Headless equivalent of this notebook:

```bash
python -m cardiosense.xray.train
```

*Research artifact. Not a medical device. Not for clinical use.*